# GC-MGT on PEMS-BAY / LargeST-GBA (Kaggle)

Re-implementation of the Granger causality-guided multi-graph transformer (Wang et al., Discover AI 2026), trained with the
**same hyperparameters as HGP** so that the two models can be compared directly: AdamW, learning rate 3e-4, weight decay 1e-4,
batch 16, up to 40 epochs, 1 warm-up epoch then cosine decay, early stopping with patience 6 on the validation mean MAE,
gradient clipping 5, Huber loss, seed 42, scored at 5 / 15 / 30 / 60 minutes. These are fixed in the code
(`gcmgt/config.py`), not in this notebook.

**One run gives the final result.** The notebook estimates the fixed Granger prior (seconds on PEMS-BAY), times a short warm run on
the GPU, trains seed 42 (set `SEEDS = (42, 7, 123)` for more) and ends with the validation and test tables. If the timing says the
seed would not fit in the session, it shrinks the number of training windows per epoch automatically, says so loudly, and records it
in every `config.json`.

**Before running**
1. Session options: Accelerator = GPU (T4 or P100, 16 GB). Internet = On if the code is cloned from GitHub.
2. Code: either push the repository and set `REPO_URL`, or add the `GCMGT` folder as a Kaggle dataset (found automatically).
3. Data, one of:
   - `DATASET = "pems_bay"` (default): add a dataset holding `pems-bay.h5` and `adj_mx_bay.pkl` (the same files the HGP PEMS-BAY
     notebook uses), or a prepared `pems_bay.npz` from `scripts/prepare_data.py`. Split 70 / 10 / 20, speed in mph, 0 = missing.
   - `DATASET = "gba"`: add the **prepared** `gba_2021.npz` that the HGP runs used (same sensors, filling and splits, so the
     comparison is on identical windows), or the raw `gba_his_raw_2021.h5` + `gba_rn_adj.npy`.
4. Press "Save & Run All". Nothing needs to be changed. Results appear under `/kaggle/working/results` (one folder per seed, in the
   HGP layout), the Granger prior under `/kaggle/working/graphs`, and the summary in `summary.json` / `summary.csv`.

In [ ]:
# ---- settings: the defaults are the final protocol, so a plain "Save & Run All" produces the final result
DATASET = "pems_bay"       # "pems_bay" (325 sensors, speed, 70/10/20) or "gba" (2,056 sensors, flow, 80/10/10)
SEEDS = (42,)              # seed 42 only (HGP's other final seeds are 7 and 123)
HORIZONS = (1, 3, 6, 12)   # scored horizons in 5-minute steps; "mean" stays the mean over 15 / 30 / 60 min
SHOW_TEST = True           # print test metrics at the end (the final runs). False while still tuning.

# training budget: None = the HGP protocol (all windows). The timing step lowers EPOCH_WINDOWS by itself only if the seeds would not fit.
EPOCH_WINDOWS = None       # random training windows per epoch (None = all)
AUTO_BUDGET = True         # let the timing step reduce EPOCH_WINDOWS if SEEDS x 40 epochs would exceed BUDGET_HOURS
BUDGET_HOURS = 10.5        # planned training time for all seeds together
SESSION_HOURS = 12.0       # Kaggle's limit; a seed is not started if it would not finish inside it
MICRO_BATCH = None         # windows per forward pass; None = 16 on PEMS-BAY, 2 on GBA. Exact gradient accumulation: speed and memory only.
TIMING_TEST = True         # time one warm epoch on 64 windows (needed for AUTO_BUDGET and the session guard)
STOP_AFTER_TIMING = False  # True = only time, do not train
GRANGER_WINDOWS = 8        # day-long windows combined into the fixed Granger prior

CODE_PATH = None           # a local folder holding gcmgt/ (testing); normally None
REPO_URL = "https://github.com/prof-sanjay/GCMGT.git"
RUN_TESTS = True

## 1. Environment

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np
import torch

T0 = time.time()      # session clock

print(sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print("warning: no GPU; only PEMS-BAY with a short budget is practical on CPU")

## 2. Code and data

In [ ]:
WORK = os.environ.get("GCMGT_WORK", "/kaggle/working")        # saved output: run folders and the Granger prior
SCRATCH = os.environ.get("GCMGT_SCRATCH", "/tmp/gcmgt")        # code, prepared data and logs: not saved
INPUT = os.environ.get("GCMGT_INPUT", "/kaggle/input")
os.makedirs(SCRATCH, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

if CODE_PATH:
    CODE = CODE_PATH
else:
    hits = glob.glob(f"{INPUT}/**/gcmgt/config.py", recursive=True)
    if hits:
        CODE = os.path.dirname(os.path.dirname(hits[0]))
    elif REPO_URL:
        CODE = f"{SCRATCH}/gcmgt_code"
        shutil.rmtree(CODE, ignore_errors=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
        found = glob.glob(f"{CODE}/**/gcmgt/config.py", recursive=True)
        assert found, "the cloned repository has no gcmgt/config.py"
        CODE = os.path.dirname(os.path.dirname(found[0]))
    else:
        raise SystemExit("GC-MGT code not found: add the repository as a Kaggle dataset or set REPO_URL")
print("code:", CODE)

files = sorted(p for p in glob.glob(f"{INPUT}/**/*", recursive=True) if os.path.isfile(p))
base = lambda p: os.path.basename(p).lower()


def inventory():
    print(f"files under {INPUT}: {len(files)}")
    for p in files[:40]:
        print(f"  {os.path.getsize(p) / 1e6:10.1f} MB  {p}")
    if len(files) > 40:
        print(f"  ... and {len(files) - 40} more")


def npz_keys(p):
    try:
        with np.load(p, allow_pickle=False) as z:
            return set(z.files)
    except Exception:
        return set()


def has_h5_key(p, key):
    try:
        import h5py
        with h5py.File(p, "r") as f:
            return key in f
    except Exception:
        return False


def need(message):
    inventory()
    raise SystemExit(message + " Attach the dataset in the notebook's right-hand panel (Add Input), then re-run.")


PREPARED = None
if DATASET == "gba":
    # content, not names: a prepared file holds flow / inputs / road; a raw GBA file holds a "traffic" table
    prepared = [p for p in files if base(p).endswith(".npz") and {"flow", "inputs", "road", "timestamps"} <= npz_keys(p)]
    traffic = [p for p in files if base(p).endswith(".h5") and has_h5_key(p, "traffic")]
    adjacency = [p for p in files if base(p).endswith(".npy") and "adj" in base(p)]
    if prepared:
        PREPARED = prepared[0]
        print("prepared data:", PREPARED, "(the file the HGP runs used)")
    elif traffic and adjacency:
        TRAFFIC, ADJACENCY = traffic[0], adjacency[0]
        print("traffic:", TRAFFIC)
        print("adjacency:", ADJACENCY)
        print("warning: raw files keep all 2,352 sensors; HGP used the 2,056 left after the duplicate filter")
    else:
        need("No GBA data found: expected a prepared gba_2021.npz (keys flow, inputs, road, timestamps), or gba_his_raw_2021.h5 "
             "(table 'traffic') plus gba_rn_adj.npy.")
elif DATASET == "pems_bay":
    traffic = sorted((p for p in files if base(p).endswith(".h5") and "meta" not in base(p) and not has_h5_key(p, "traffic")),
                     key=lambda p: "pems" not in base(p))                 # files named pems-bay first
    prepared = [p for p in files if base(p).endswith(".npz") and {"flow", "inputs", "road", "timestamps"} <= npz_keys(p)]
    adjacency = [p for p in files if base(p).endswith(".pkl") and "adj" in base(p)]
    if prepared:
        PREPARED = prepared[0]
        print("prepared data:", PREPARED)
    elif traffic and adjacency:
        TRAFFIC, ADJACENCY = traffic[0], adjacency[0]
        print("traffic:", TRAFFIC)
        print("adjacency:", ADJACENCY)
    else:
        need("No PEMS-BAY data found: expected pems-bay.h5 and adj_mx_bay.pkl, or a prepared pems_bay.npz.")
else:
    raise SystemExit('DATASET must be "gba" or "pems_bay"')

In [ ]:
def run(cmd, keep=None):
    """Run a command in the code folder; print only the lines for which keep(line) is true. The full log goes to a file."""
    name = os.path.basename(cmd[1]) if cmd[1] != "-m" else cmd[2]
    log = open(f"{SCRATCH}/log_{name}.txt", "a")
    with subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            log.write(line)
            if keep is None or keep(line):
                print(line, end="")
    log.close()
    assert proc.returncode == 0, f"{name} exited with {proc.returncode}; see {SCRATCH}/log_{name}.txt"


def last_line(line):
    return " passed" in line or " failed" in line or "error" in line.lower()


def prior_lines(line):
    return line.startswith(("estimating", "  granger window", "Traceback")) or "edges" in line


def train_lines(line):
    s = line.strip()
    if s.startswith("epoch"):
        return int(s.split()[1]) % 5 == 0 or int(s.split()[1]) == 1
    return SHOW_TEST and s.startswith("test")


MICRO = MICRO_BATCH or (16 if DATASET == "pems_bay" else 2)


if RUN_TESTS:
    run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "-W", "ignore"], keep=last_line)

## 3. Prepare the data
Skipped when a prepared file was found. Otherwise the same `scripts/prepare_data.py` as HGP writes it.

In [ ]:
if PREPARED:
    DATA = PREPARED
else:
    DATA = f"{SCRATCH}/data/{DATASET}.npz"
    if not os.path.exists(DATA):
        run([sys.executable, "scripts/prepare_data.py", "--dataset", DATASET, "--traffic", TRAFFIC, "--adjacency", ADJACENCY, "--out", DATA])
print("data:", DATA)

## 4. Fixed Granger prior
Pairwise Granger F-tests with Benjamini-Hochberg control inside each day-long window, estimated once from the training split and cached in `graphs/`. The regressions run on the CPU (about 2 minutes per window at 2,056 sensors on a laptop), so expect roughly 20 minutes for the default 8 windows; a re-run reuses the cache.

In [ ]:
GRAPHS = f"{WORK}/graphs"
run([sys.executable, "scripts/build_prior.py", "--data", DATA, "--graphs", GRAPHS, "--granger-windows", str(GRANGER_WINDOWS)], keep=prior_lines)

## 5. Timing and budget
Trains two epochs on 64 windows (the second one is warm) and extrapolates to the full protocol of 40 epochs per seed. If SEEDS x 40 epochs would not fit in `BUDGET_HOURS`, `EPOCH_WINDOWS` is lowered to fit; the choice is printed and written into every run's `config.json`.

In [ ]:
SEED_SECONDS = None
if TIMING_TEST:
    sys.path.insert(0, CODE)
    from gcmgt.config import DataConfig
    from gcmgt.data import Windows, load

    dc = DataConfig.from_file(DATA)
    data = load(dc)
    steps = tuple(range(1, dc.out_steps + 1))
    n_train, n_val = (len(Windows(s, dc.seq_len, steps)) for s in (data.train, data.val))
    del data
    probe = f"{SCRATCH}/timing"
    shutil.rmtree(probe, ignore_errors=True)
    run([sys.executable, "scripts/train.py", "--data", DATA, "--graphs", GRAPHS, "--results", probe, "--granger-windows", str(GRANGER_WINDOWS),
         "--epochs", "2", "--max-windows", "64", "--micro-batch", str(MICRO)], keep=lambda l: l.strip().startswith("epoch"))
    sec = float(open(f"{probe}/gcmgt__seed42/history.csv").read().splitlines()[-1].split(",")[-1])     # the warm (second) epoch
    per_window = 1.25 * sec / 128                             # 64 training + 64 validation windows, plus a 25 % margin (a local test ran ~1.4x slower than timed)
    n_epoch = min(EPOCH_WINDOWS or n_train, n_train)
    seed_s = 40 * per_window * (n_epoch + n_val)
    print(f"windows: train {n_train:,}  validation {n_val:,}  | micro-batch {MICRO}")
    print(f"estimated {per_window * (n_epoch + n_val) / 60:.1f} min per epoch, {seed_s / 3600:.2f} h per seed, "
          f"{len(SEEDS) * seed_s / 3600:.2f} h for {len(SEEDS)} seeds (early stopping may end sooner)")
    if AUTO_BUDGET and len(SEEDS) * seed_s > BUDGET_HOURS * 3600:
        allowed = BUDGET_HOURS * 3600 / len(SEEDS) / 40 / per_window - n_val
        EPOCH_WINDOWS = int(max(1024, allowed // 16 * 16))
        seed_s = 40 * per_window * (min(EPOCH_WINDOWS, n_train) + n_val)
        print("=" * 100)
        print(f"BUDGET REDUCED: {len(SEEDS)} full seeds would not fit in {BUDGET_HOURS} h, so each epoch now uses {EPOCH_WINDOWS:,} random training windows "
              f"of {n_train:,}.")
        print("This differs from the HGP protocol (all windows per epoch): report it. New estimate: "
              f"{seed_s / 3600:.2f} h per seed, {len(SEEDS) * seed_s / 3600:.2f} h in total.")
        print("=" * 100)
    else:
        print("The full protocol fits: every epoch uses all training windows.")
    SEED_SECONDS = seed_s
    if STOP_AFTER_TIMING:
        print("STOP_AFTER_TIMING is True: not training")

## 6. Train and evaluate
One run per seed. Each writes `results.json`, `history.csv`, `config.json`, `checkpoints/best.pt` and `test_predictions_h<h>.npz` into `results/gcmgt__seed<seed>/` (the same layout as the HGP runs, so paired tests work across the two models).

In [ ]:
RESULTS = f"{WORK}/results"
if TIMING_TEST and STOP_AFTER_TIMING:
    print("skipped: STOP_AFTER_TIMING is True")
else:
    for seed in SEEDS:
        if os.path.exists(f"{RESULTS}/gcmgt__seed{seed}/results.json"):
            print("already done: seed", seed)
            continue
        elapsed = time.time() - T0
        if SEED_SECONDS and elapsed + SEED_SECONDS > SESSION_HOURS * 3600 - 1800:
            print(f"NOT STARTED: seed {seed}. {elapsed / 3600:.1f} h used and one seed needs about {SEED_SECONDS / 3600:.1f} h; the session limit is {SESSION_HOURS} h.")
            continue
        print(f"--- seed {seed}  ({elapsed / 3600:.2f} h into the session)")
        cmd = [sys.executable, "scripts/train.py", "--seed", str(seed), "--data", DATA, "--graphs", GRAPHS, "--results", RESULTS,
               "--granger-windows", str(GRANGER_WINDOWS), "--horizons", *map(str, HORIZONS), "--micro-batch", str(MICRO)]
        if EPOCH_WINDOWS:
            cmd += ["--epoch-windows", str(EPOCH_WINDOWS)]
        limit = max(0.25, SESSION_HOURS - 0.75 - elapsed / 3600)        # hard stop: results are always written before Kaggle's limit
        cmd += ["--time-limit-hours", f"{limit:.3f}"]
        run(cmd, keep=train_lines)
        r = json.load(open(f"{RESULTS}/gcmgt__seed{seed}/results.json"))
        print(f"done: best epoch {r['best_epoch']} of {r['epochs']} run (max 40), stopped by {r['stop_reason']}, "
              f"best val MAE {r['validation']['mean']['mae']:.3f}, {r['training_seconds'] / 60:.1f} min")
        if r["stop_reason"] == "time_limit":
            print("WARNING: stopped by the time limit before early stopping or 40 epochs; the cosine schedule did not finish. Report it.")

## 7. Results
Validation per run, then (with `SHOW_TEST = True`) test metrics per seed and the mean and standard deviation over seeds (sample standard deviation). The recipe actually used is printed from `config.json` so it can be checked against HGP. `summary.json` and `summary.csv` are written next to `results/`.

In [ ]:
import csv

runs = [json.load(open(f)) for f in sorted(glob.glob(f"{RESULTS}/*/results.json"))]
assert runs, "no finished runs under " + RESULTS

print("VALIDATION")
print(f"{'run':16s} {'valMAE':>7s} {'15min':>6s} {'30min':>6s} {'60min':>6s} {'valRMSE':>8s} {'best':>4s} {'min':>6s}")
for r in runs:
    v = r["validation"]
    print(f"{r['run']:16s} {v['mean']['mae']:7.3f} {v['3']['mae']:6.3f} {v['6']['mae']:6.3f} {v['12']['mae']:6.3f} {v['mean']['rmse']:8.3f} "
          f"{r['best_epoch']:4d} {r['training_seconds'] / 60:6.1f}   stopped by {r.get('stop_reason', '?')} after {r['epochs']} epochs")

cfgs = sorted(glob.glob(f"{RESULTS}/*/config.json"))
t = json.load(open(cfgs[0]))["train"]
keys = ("optimizer", "learning_rate", "weight_decay", "batch_size", "micro_batch", "epochs", "schedule", "warmup_epochs",
        "patience", "grad_clip", "loss", "huber_delta", "epoch_windows")
print("\nrecipe used:", {k: t[k] for k in keys})
if t["epoch_windows"]:
    print(f"NOTE: epoch_windows = {t['epoch_windows']:,}: each epoch used a random subset of the training windows (budget reduced); report it.")

if SHOW_TEST:
    cols = [(f"{h * 5} min", str(h)) for h in HORIZONS] + [("mean 15/30/60", "mean")]
    print("\nTEST per seed (MAE / RMSE / MAPE %)")
    for r in runs:
        print(r["run"])
        for label, k in cols:
            m = r["test"][k]
            print(f"   {label:14s} MAE {m['mae']:8.3f}  RMSE {m['rmse']:8.3f}  MAPE {m['mape']:7.3f}")
    print(f"\nTEST mean +- std over {len(runs)} seed(s)")
    rows = []
    for label, k in cols:
        vals = {q: np.array([r["test"][k][q] for r in runs]) for q in ("mae", "rmse", "mape")}
        sd = {q: (float(vals[q].std(ddof=1)) if len(runs) > 1 else None) for q in vals}
        f3 = lambda x: "   n/a" if x is None else f"{x:6.3f}"
        print(f"   {label:14s} MAE {vals['mae'].mean():8.3f} +- {f3(sd['mae'])}   RMSE {vals['rmse'].mean():8.3f} +- {f3(sd['rmse'])}   "
              f"MAPE {vals['mape'].mean():7.3f} +- {f3(sd['mape'])}")
        rows.append({"horizon": label, **{f"{q}_mean": float(vals[q].mean()) for q in vals}, **{f"{q}_std": sd[q] for q in vals}})
    json.dump({"seeds": [r["run"] for r in runs], "recipe": {k: t[k] for k in keys}, "test": rows}, open(f"{WORK}/summary.json", "w"), indent=1)
    with open(f"{WORK}/summary.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    print(f"\nwrote {WORK}/summary.json and summary.csv")